# Непрерывный фильтр и укрупнение шага наблюдений

Лёгкий пример: два состояния $\theta$, две координаты $Y$, интенсивности переходов $0.4$ и $0.3$, горизонт $T=4$. При скачке метка переразыгрывается из **треугольного** распределения на носителе нового состояния. Наблюдения **гауссовские**:
$$dX_t^k=Y_t^k\,dt+0.7\,dW_t^k,\qquad k=1,2.$$
Сетки разных состояний сдвинуты; на каждой по $6\times6$ узлов. Все фильтры получают одинаковую нормированную плотность и одинаковые сетки.

Для каждого из трёх фиксированных семян наблюдение один раз генерируется на сетке $h_{obs}=0.001$, включая $T$. При укрупнении складываются его приращения: новый шум не генерируется. Непрерывный фильтр использует шаги $0.02$ и $0.01$, дискретный — $0.5,0.25,0.1,0.05$. В качестве опоры ниже взят шаг $0.01$; различие с $0.02$ характеризует чувствительность опоры к временному шагу.

In [ ]:
import _bootstrap  # noqa: F401

import time
import numpy as np
import numba as nb
import matplotlib.pyplot as plt

from discretized_filter.config import set_config
from discretized_filter.core.filter import Filter
from discretized_filter.core.filter_continuous import ContinuousFilter
from discretized_filter.core.smjp import sparse_mc
from discretized_filter.utils.grids import set_seed

cfg = set_config('toy_continuous_additive')
# Нормируем один раз до создания любого фильтра.
cfg.pi = cfg.pi / (cfg.pi.sum(axis=1) * cfg.delta)[:, None]
cfg.pi_init = cfg.p0[:, None] * cfg.pi

h_obs = 0.001
h_continuous = 0.02
h_reference = 0.01
discrete_steps = np.array([0.5, 0.25, 0.1, 0.05])
seeds = (20260906, 20260907, 20260908)
common_times = np.arange(0.5, cfg.T + 0.25, 0.5)
snapshot_times = (1.0, 2.0, 4.0)

def time_grid(step):
    count = int(round(cfg.T / step))
    assert np.isclose(count * step, cfg.T)
    return np.linspace(0.0, cfg.T, count + 1)

print(f'Состояний: {cfg.N}; узлов на состояние: {cfg.M_net.shape[1]}; T={cfg.T:g}')
print(f'Наблюдения: {h_obs:g}; опора: {h_reference:g}; сравнение: {common_times}')

## Расчёт на общей траектории

Используется специализированное гауссовское ядро дискретного фильтра, не более одного скачка за шаг и два узла квадратуры времени скачка. Поэтому измеряемое расхождение включает потерю информации при агрегировании наблюдений и ошибки временных схем. Пространственная сетка фиксирована: этот опыт не проверяет сходимость по пространству.

Все ошибки считаются в одни и те же моменты $0.5,1,\ldots,4$. Метрики: $\sum_{n,j}|\psi_{n,j}-\psi^{ref}_{n,j}|\Delta_n$, $L^1$ вероятностей состояний и евклидова норма ошибки среднего $Y$. Затем берётся среднее по времени и трём семенам.

In [ ]:
def run_filter(observations, step, continuous):
    sampled = observations.sample(time_grid(step), include_events=True)
    if continuous:
        filt = ContinuousFilter.from_config(cfg, ht=step)
    else:
        filt = Filter(
            cfg.pi_init, cfg.pi, cfg.M_net, cfg.C,
            cfg.N, cfg.Lambda, step, cfg.delta, cfg.obs_density,
            n_points=cfg.n_points, two_jumps=cfg.two_jumps,
            filter_step=cfg.filter_step,
        )
    densities = [filt.psi.copy()]
    theta_est, y_est = filt.estimate()
    probabilities, means = [theta_est], [y_est]
    for dt, increment in zip(np.diff(sampled.times), sampled.increments):
        if continuous:
            filt.update(increment, dt=float(dt))
        else:
            assert np.isclose(dt, step)
            filt.update(increment)
        assert np.isfinite(filt.psi).all()
        assert np.min(filt.psi) >= -1e-14
        assert np.isclose(np.sum(filt.psi * cfg.delta[:, None]), 1.0)
        theta_est, y_est = filt.estimate()
        assert np.isclose(theta_est.sum(), 1.0)
        densities.append(filt.psi.copy())
        probabilities.append(theta_est)
        means.append(y_est)
    return dict(times=sampled.times, psi=np.array(densities),
                theta=np.array(probabilities), y=np.array(means))

def common_indices(run, times=common_times):
    # linspace может дать соседний узел из-за округления.
    indices = np.array([np.argmin(abs(run['times'] - t)) for t in times])
    assert np.allclose(run['times'][indices], times, rtol=0, atol=1e-10)
    return indices

def errors(run, reference):
    i, j = common_indices(run), common_indices(reference)
    return np.column_stack((
        np.sum(abs(run['psi'][i] - reference['psi'][j]) * cfg.delta[None, :, None],
               axis=(1, 2)),
        np.sum(abs(run['theta'][i] - reference['theta'][j]), axis=1),
        np.linalg.norm(run['y'][i] - reference['y'][j], axis=1),
    ))

started = time.perf_counter()
previous_threads = nb.get_num_threads()
runs, error_runs, refinement_runs = {}, [], []
try:
    nb.set_num_threads(min(2, previous_threads))
    for seed in seeds:
        set_seed(seed)
        theta, y, jump_ends = sparse_mc(
            cfg.p0, cfg.Lambda, cfg.lam, cfg.T, cfg.get_y, cfg.y_intervals,
        )
        observations = cfg.get_continuous_obs(
            time_grid(h_obs), theta, y, jump_ends, seed=seed + 1000,
        )
        # Проверяем, что агрегирование сохраняет полное приращение.
        total = observations.increments.sum(axis=0)
        for step in (*discrete_steps, h_continuous, h_reference):
            assert np.allclose(observations.sample(time_grid(step)).increments.sum(axis=0), total)
        reference = run_filter(observations, h_reference, True)
        continuous = run_filter(observations, h_continuous, True)
        discrete = {float(h): run_filter(observations, float(h), False)
                    for h in discrete_steps}
        error_runs.append(np.array([errors(discrete[float(h)], reference).mean(axis=0)
                                    for h in discrete_steps]))
        refinement_runs.append(errors(continuous, reference).mean(axis=0))
        runs[seed] = dict(theta=theta, y=y, jump_ends=jump_ends,
                          reference=reference, continuous=continuous, discrete=discrete)
        print(f'Семя {seed}: скачков {len(theta) - 1}')
finally:
    nb.set_num_threads(previous_threads)

error_runs = np.array(error_runs)
refinement_runs = np.array(refinement_runs)
mean_errors = error_runs.mean(axis=0)
mean_refinement = refinement_runs.mean(axis=0)
elapsed = time.perf_counter() - started
print(f'Время расчёта: {elapsed:.2f} с (включая компиляцию при первом запуске)')
print('Шаг      L1 плотности     L1 theta        Ошибка Y')
for step, row in zip(discrete_steps, mean_errors):
    print(f'{step:5.2f}    {row[0]:12.6g}    {row[1]:12.6g}    {row[2]:12.6g}')
print('Опора .02/.01:', mean_refinement)
print('Уточнение / ошибка при h=.05:', mean_refinement / mean_errors[-1])

In [ ]:
metric_names = ('Плотность: L¹', 'Вероятности θ: L¹', 'Среднее Y: норма ошибки')
fig, axes = plt.subplots(1, 3, figsize=(11, 3.1))
order = np.argsort(discrete_steps)
for k, ax in enumerate(axes):
    for values in error_runs:
        ax.loglog(discrete_steps[order], values[order, k], color='C0', alpha=0.22, lw=1)
    ax.loglog(discrete_steps[order], mean_errors[order, k], 'o-', color='C0', label='Дискретный: среднее')
    ax.axhline(mean_refinement[k], color='0.25', ls='--', label='Опора: 0.02 / 0.01')
    ax.set(xlabel='Шаг дискретного фильтра', title=metric_names[k])
    ax.set_xticks(discrete_steps[order], [f'{h:g}' for h in discrete_steps[order]])
    ax.set_xticks([], minor=True)
    ax.grid(alpha=0.2)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', ncol=2, frameon=False)
fig.tight_layout(rect=(0, 0, 1, 0.9))
plt.show()

Тонкие линии соответствуют отдельным семенам. Убывание средней ошибки на этой конечной сетке — численная иллюстрация, а не доказательство сходимости. Монотонность для каждого отдельного наблюдения не предполагается. Отношение «уточнение / ошибка при $h=0.05$» показывает, насколько временная погрешность опоры мала относительно наиболее точного дискретного сравнения; оно не является строгой оценкой остаточной ошибки.

In [ ]:
example = runs[seeds[0]]
ref = example['reference']
coarse = example['discrete'][0.5]
fine = example['discrete'][0.05]
starts = np.r_[0.0, example['jump_ends'][:-1]]

fig, axes = plt.subplots(3, 1, figsize=(10, 6), sharex=True)
axes[0].step(np.r_[starts, cfg.T], np.r_[example['theta'], example['theta'][-1]],
             where='post', color='0.7', label='Истинное θ')
axes[0].plot(ref['times'], ref['theta'][:, 1], color='C0', label='P(θ=1): опора')
axes[0].plot(fine['times'], fine['theta'][:, 1], '--', color='C1', label='P(θ=1): h=0.05')
axes[0].set(ylabel='θ / вероятность', ylim=(-0.05, 1.05))
for dim, ax in enumerate(axes[1:]):
    ax.step(np.r_[starts, cfg.T], np.r_[example['y'][:, dim], example['y'][-1, dim]],
            where='post', color='0.7', label='Истинное Y')
    ax.plot(ref['times'], ref['y'][:, dim], color='C0', label='Опора')
    ax.plot(coarse['times'], coarse['y'][:, dim], ':', color='C2', label='h=0.5')
    ax.plot(fine['times'], fine['y'][:, dim], '--', color='C1', label='h=0.05')
    ax.set(ylabel=f'Y{dim + 1}')
    ax.grid(alpha=0.2)
axes[0].legend(loc='upper left', bbox_to_anchor=(1.01, 1), frameon=False)
axes[1].legend(loc='upper left', bbox_to_anchor=(1.01, 1), frameon=False)
axes[-1].set_xlabel('Время')
fig.tight_layout()
plt.show()

## Плотности в трёх моментах

Для маргинальной плотности $Y^k$ интегрируем вторую координату с её пространственным шагом и складываем состояния **по фактическим координатам их узлов**. Нельзя просто сложить одинаковые индексы сдвинутых сеток. Совместные плотности показаны на двух соответствующих прямоугольниках с одной цветовой шкалой для всех моментов и методов.

In [ ]:
def marginal_density(psi, dim):
    coordinates = np.unique(cfg.M_net[:, :, dim])
    density = np.zeros(coordinates.size)
    for state in range(cfg.N):
        local = np.unique(cfg.M_net[state, :, dim])
        spacing = local[1] - local[0]
        index = np.searchsorted(coordinates, cfg.M_net[state, :, dim])
        np.add.at(density, index, psi[state] * cfg.delta[state] / spacing)
    return coordinates, density

methods = ((ref, 'Опора 0.01', 'C0', '-'),
           (coarse, 'h=0.5', 'C2', ':'),
           (fine, 'h=0.05', 'C1', '--'))
fig, axes = plt.subplots(2, 3, figsize=(10, 5))
for column, t in enumerate(snapshot_times):
    for dim in range(2):
        ax = axes[dim, column]
        for run, label, color, style in methods:
            psi = run['psi'][common_indices(run, [t])[0]]
            x, density = marginal_density(psi, dim)
            ax.plot(x, density, color=color, ls=style, lw=1.8, label=label)
        ax.set(xlabel=f'Y{dim + 1}', title=f't={t:g}')
        if column == 0:
            ax.set_ylabel('Плотность')
        ax.grid(alpha=0.2)
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', ncol=3, frameon=False)
fig.tight_layout(rect=(0, 0, 1, 0.91))
plt.show()

snapshots = [[run['psi'][common_indices(run, [t])[0]] for run, *_ in methods]
             for t in snapshot_times]
vmax = max(psi.max() for row in snapshots for psi in row)
fig, axes = plt.subplots(3, 3, figsize=(8, 7), sharex=True, sharey=True,
                         layout='constrained')
for row, t in enumerate(snapshot_times):
    for column, (_, label, _, _) in enumerate(methods):
        ax = axes[row, column]
        for state in range(cfg.N):
            x = np.unique(cfg.M_net[state, :, 0])
            y = np.unique(cfg.M_net[state, :, 1])
            # Раскладываем по координатам, независимо от порядка декартовой сетки.
            z = np.zeros((len(y), len(x)))
            ix = np.searchsorted(x, cfg.M_net[state, :, 0])
            iy = np.searchsorted(y, cfg.M_net[state, :, 1])
            z[iy, ix] += snapshots[row][column][state]
            mesh = ax.pcolormesh(x, y, z, shading='nearest', cmap='viridis',
                                 vmin=0.0, vmax=vmax)
        ax.set_title(f'{label}, t={t:g}')
        if row == 2:
            ax.set_xlabel('Y1')
        if column == 0:
            ax.set_ylabel('Y2')
fig.colorbar(mesh, ax=axes, label='Совместная плотность Y', shrink=0.8)
plt.show()